In [1]:
# Core scverse libraries
import scanpy as sc
import anndata as ad
import pandas as pd
import matplotlib.pyplot as plt
import scanpy.external as sce
import json

In [2]:
sc.settings.verbosity = 3  # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.settings.set_figure_params(
    dpi=300,
    facecolor="white",
    figsize=(10, 8),  # Adjust as needed
    fontsize=22
)

plt.rcParams['axes.grid'] = False

/Users/takahiro/miniforge3/envs/scanpy/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


scanpy==1.10.3 anndata==0.9.2 umap==0.5.6 numpy==1.26.4 scipy==1.12.0 pandas==2.1.4 scikit-learn==1.5.2 statsmodels==0.14.4 igraph==0.11.6 pynndescent==0.5.12


In [3]:
adata1 = sc.read_10x_h5("/Users/takahiro/Desktop/project/Reha/cellbender/Fhl2OE/Fhl2OE1_cellbender_filtered.h5")
adata2 = sc.read_10x_h5("/Users/takahiro/Desktop/project/Reha/cellbender/Fhl2OE/Fhl2OE2_cellbender_filtered.h5")

reading /Users/takahiro/Desktop/project/Reha/cellbender/Fhl2OE/Fhl2OE1_cellbender_filtered.h5
 (0:00:00)
reading /Users/takahiro/Desktop/project/Reha/cellbender/Fhl2OE/Fhl2OE2_cellbender_filtered.h5


/Users/takahiro/miniforge3/envs/scanpy/lib/python3.10/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Users/takahiro/miniforge3/envs/scanpy/lib/python3.10/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


 (0:00:00)


/Users/takahiro/miniforge3/envs/scanpy/lib/python3.10/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Users/takahiro/miniforge3/envs/scanpy/lib/python3.10/site-packages/anndata/_core/anndata.py:1840: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [4]:
# 変数名をユニークにする
for adata in [adata1, adata2]:
    adata.var_names_make_unique()


In [5]:
# サンプル名リスト
sample_names = ["Fhl2OE1", "Fhl2OE2"]

# 変数名をユニークにし、サンプル名を追加する
adata_list = [adata1, adata2]
for adata, sample_name in zip(adata_list, sample_names):
    adata.var_names_make_unique()
    adata.obs["sample"] = sample_name

# AnnData オブジェクトを結合
adata = sc.concat(adata_list)

# 結果を確認
print(adata)

AnnData object with n_obs × n_vars = 21050 × 32285
    obs: 'sample'


/Users/takahiro/miniforge3/envs/scanpy/lib/python3.10/site-packages/anndata/_core/anndata.py:1838: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [6]:
adata.obs["sample"].unique()

array(['Fhl2OE1', 'Fhl2OE2'], dtype=object)

In [7]:
adata.obs["type"] = adata.obs["sample"].map({
    "Fhl2OE1": "Fhl2OE",
    "Fhl2OE2": "Fhl2OE",
})

In [8]:
adata.obs_names_make_unique()

In [9]:
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

filtered out 308 cells that have less than 200 genes expressed
filtered out 9929 genes that are detected in less than 3 cells


In [10]:
# annotate the group of mitochondrial genes as "mt"
adata.var["mt"] = adata.var_names.str.startswith("mt-")
sc.pp.calculate_qc_metrics(
    adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True
)

In [11]:
sc.pl.highest_expr_genes(adata, n_top=20)

normalizing counts per cell
    finished (0:00:00)


In [12]:
# for figure
# Create a figure with subplots
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Plotting the individual graphs
sc.pl.violin(adata, ['n_genes_by_counts'], groupby='sample', ax=axes[0], show=False)
sc.pl.violin(adata, ['total_counts'], groupby='sample', ax=axes[1], show=False)
sc.pl.violin(adata, ['pct_counts_mt'], groupby='sample', ax=axes[2], show=False)

# Adjusting layout
plt.tight_layout()

# Display the plot
plt.show()

In [13]:
sc.pl.violin(
    adata,
    ["pct_counts_mt"],
    jitter=0,
    multi_panel=True,
)

In [14]:
sc.pl.scatter(adata, x="total_counts", y="pct_counts_mt")
sc.pl.scatter(adata, x="total_counts", y="n_genes_by_counts")

In [15]:
adata = adata[(adata.obs.n_genes_by_counts < 6000), :]
adata = adata[adata.obs.pct_counts_mt < 1, :].copy()

In [19]:
sc.pp.scrublet(adata, batch_key="sample")

Running Scrublet
filtered out 1930 genes that are detected in less than 3 cells
normalizing counts per cell
    finished (0:00:00)
extracting highly variable genes
    finished (0:00:00)
--> added
    'highly_variable', boolean vector (adata.var)
    'means', float vector (adata.var)
    'dispersions', float vector (adata.var)
    'dispersions_norm', float vector (adata.var)
normalizing counts per cell
    finished (0:00:00)
normalizing counts per cell
    finished (0:00:00)
Embedding transcriptomes using PCA...
    using data matrix X directly
Automatically set threshold at doublet score = 0.24
Detected doublet rate = 2.4%
Estimated detectable doublet fraction = 34.0%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 7.1%
filtered out 1470 genes that are detected in less than 3 cells
normalizing counts per cell
    finished (0:00:00)
extracting highly variable genes
    finished (0:00:00)
--> added
    'highly_variable', boolean vector (adata.var)
    'means', float vector (adata

In [20]:
adata.obs["doublet_score"].hist(bins=100)

<Axes: >

In [21]:
adata = adata[~adata.obs["predicted_doublet"]]

In [22]:
adata = adata[adata.obs["doublet_score"]<0.1,]

In [ ]:
adata.raw = adata.copy()

In [16]:
sc.pp.normalize_total(adata, target_sum=1e4)

normalizing counts per cell
    finished (0:00:00)


In [17]:
sc.pp.log1p(adata)

In [19]:
sc.pp.highly_variable_genes(adata, n_top_genes=2000)

extracting highly variable genes
    finished (0:00:00)
--> added
    'highly_variable', boolean vector (adata.var)
    'means', float vector (adata.var)
    'dispersions', float vector (adata.var)
    'dispersions_norm', float vector (adata.var)


In [20]:
sc.pl.highly_variable_genes(adata)

In [21]:
adata = adata[:,adata.var["highly_variable"]]

In [22]:
sc.pp.scale(adata, max_value=10)

... as `zero_center=True`, sparse input is densified and may lead to large memory consumption


/Users/takahiro/miniforge3/envs/scanpy/lib/python3.10/site-packages/scanpy/preprocessing/_scale.py:318: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)


In [ ]:
sc.tl.pca(adata, svd_solver="arpack")

computing PCA
    with n_comps=50


In [ ]:
sc.pl.pca_variance_ratio(adata, log=True)

In [ ]:
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)

In [ ]:
sc.tl.umap(adata)

In [ ]:
adata.write_h5ad("./adata/merged_processed_Fhl2OE.h5ad")

In [ ]:
adata = sc.read_h5ad("./adata/merged_processed_Fhl2OE.h5ad")

# integration

In [ ]:
sce.pp.harmony_integrate(adata, 'sample')

In [ ]:
adata

In [ ]:
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40, use_rep="X_pca_harmony")

In [ ]:
sc.tl.umap(adata)

In [ ]:
adata

In [ ]:
sc.pl.umap(adata, color=["sample","doublet_score","Mybpc3","Dcn","Cdh5","Ptprc","Pecam1","Vwf"], vmax=5)

# Remove doublets low quality cell

In [ ]:
# over clustering
sc.tl.leiden(adata, resolution=1)

In [ ]:
sc.pl.umap(adata, color=["leiden"])

In [ ]:
# for figure
# Create a figure with subplots
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Plotting the individual graphs
sc.pl.violin(adata, ['n_genes_by_counts'], groupby='leiden', ax=axes[0], show=False)
sc.pl.violin(adata, ['total_counts'], groupby='leiden', ax=axes[1], show=False)
sc.pl.violin(adata, ['pct_counts_mt'], groupby='leiden', ax=axes[2], show=False)

# Adjusting layout
plt.tight_layout()

# Display the plot
plt.show()

In [45]:
sc.pl.dotplot(adata, [
  "Ttn", "Mybpc3", #CMs
  "Dcn", "Col1a1", "Postn", # FBs
  "Frmd3", "Dlc1", "Myh11", # SMCs
  "Pecam1", "Cdh5", "Vwf","Npr3", # Enforthelial, cardialcells
  "Ptprc", "Mrc1", "Cd163", # Macrophages
  "Cd3e","Skap1", "Cd79a", "Cd79b","Il7r", "Kit", # Tcell Bcell
  "Lmnb1","Slpi","S100a9", #Granulocytes
  "Nrxn1", "Nrxn3", "Upk3b","Msln","Gpc3", 
  "Plin1", "Xkr4", "Acta2","Ms4a1","Ncr1","Vtn","Colec11","Steap4","Kcnj8","Mmrn1","Flt4"
], groupby="leiden", vmax=5)

In [46]:
# cluster 11 20 22 potencial doublets
adata = adata[~adata.obs["leiden"].isin(["15","21"])]

# Cell type annotation

In [47]:
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=20, use_rep="X_pca_harmony")

computing neighbors
    finished: added to `.uns['neighbors']`
    `.obsp['distances']`, distances for each pair of neighbors
    `.obsp['connectivities']`, weighted adjacency matrix (0:00:01)


In [48]:
sc.tl.umap(adata)

computing UMAP
    finished: added
    'X_umap', UMAP coordinates (adata.obsm)
    'umap', UMAP parameters (adata.uns) (0:00:08)


In [49]:
# over clustering
sc.tl.leiden(adata, resolution=0.5)

running Leiden clustering
    finished: found 15 clusters and added
    'leiden', the cluster labels (adata.obs, categorical) (0:00:00)


In [50]:
sc.pl.umap(adata, color=["leiden"], label=True)

In [52]:
sc.pl.dotplot(adata, [
  "Ttn", "Mybpc3", #CMs
  "Dcn", "Col1a1", "Postn", # FBs
  "Pdgfrb", "Dlc1", "Myh11", # SMCs
  "Pecam1", "Cdh5", "Vwf","Npr3", # Enforthelial, cardialcells
  "Ptprc", "Mrc1", "Cd163", # Macrophages
  "Cd3e","Skap1", "Cd79a", "Cd79b","Il7r", "Kit", # Tcell Bcell
  "Lmnb1","Slpi","S100a9", #Granulocytes
  "Nrxn1", "Nrxn3", "Upk3b","Msln","Gpc3", 
  "Plin1", "Xkr4", "Acta2","Ms4a1","Ncr1","Vtn","Colec11","Steap4","Kcnj8","Mmrn1","Flt4"
], groupby="leiden", vmax=5)

In [55]:
# 各細胞タイプに対応するクラスタ番号の辞書
cell_type_dict = {
    "Cardiomyocytes": ["1", "3", "13"],
    "Fibroblasts": ["0"],
    "Endothelial Cells": ["2", "6"],
    "Myeloid Cells": ["4","8"],
    "Pericytes": ["5","12"],
    "Endocardial Cells": ["7"],
    "Lympatic Endothelial Cells": ["9"],
    "Epicardial Cells": ["14"],
    "B cells": ["10"],
    "Lymphoid Cells": ["11"]
}

# クラスタ番号と細胞タイプのマッピングを作成
leiden_to_cell_type = {cluster: cell_type for cell_type, clusters in cell_type_dict.items() for cluster in clusters}

# adata.obs["cell_type"]にマッピングを適用
adata.obs["cell_type"] = adata.obs["leiden"].map(leiden_to_cell_type)


In [56]:
sc.settings.set_figure_params(dpi=300, facecolor="white", figsize=(8, 8))
sc.pl.umap(adata, color=["cell_type"], size=10, save="umap.tiff")

In [59]:
cell_type_order = [
    "Cardiomyocytes", 
    "Fibroblasts", 
    "Endothelial Cells", 
    "Lympatic Endothelial Cells", 
    "Endocardial Cells", 
    "Epicardial Cells", 
    "Pericytes",  
    "Myeloid Cells", 
    "Lymphoid Cells",
    "B cells"
]

sc.pl.dotplot(adata, [
  "Ttn", "Mybpc3", #CMs
  "Dcn", "Col1a1", # FBs
  "Pecam1", "Cdh5", "Mmrn1","Flt4", # Endothelial Lymphtatic
  "Vwf","Npr3", # Endocardialcells
  "Msln","Gpc3", # Epicardial
  "Pdgfrb", "Nrxn1", "Myh11", # Pericytes SMCs
  "Ptprc", "Mrc1", "Cd163", # Macrophages
  "Skap1", "Cd79a", "Cd79b" # Tcell Bcell
], groupby="cell_type", vmax=5, categories_order=cell_type_order)

In [63]:
adata.write_h5ad("./adata/allpopulation_curated_Fhl2.h5ad")

In [64]:
adata_full = adata.raw.to_adata().copy()

In [65]:
adata_full.write_h5ad("./adata/rawadata_curated_Fhl2.h5ad")

In [66]:
adatacm = adata_full[adata_full.obs["cell_type"]=="Cardiomyocytes"]
adatafb = adata_full[adata_full.obs["cell_type"]=="Fibroblasts"]
adataec = adata_full[adata_full.obs["cell_type"]=="Endothelial Cells"]
adatamye = adata_full[adata_full.obs["cell_type"]=="Myeloid Cells"]
adataendo = adata_full[adata_full.obs["cell_type"]=="Endocardial Cells"]
adataepi = adata_full[adata_full.obs["cell_type"]=="Epicardial Cells"]
adatalymec = adata_full[adata_full.obs["cell_type"]=="Lympatic Endothelial Cells"]

In [67]:
adatacm.write_h5ad("./adata/CM_Fhl2OE.h5ad")